In [32]:
import pandas as pd
from fastparquet import write
from fastparquet import ParquetFile
from pathlib import Path
import numpy as np
import os

In [ ]:
data_pipeline = "round"
input_pipeline = "baseline"
place = 2

In [34]:
if os.environ.get('KAGGLE_KERNEL_RUN_TYPE'):
    data_path = "../../kaggle/input/datasets/abhinavneelam/smartphone-addiction/data/"
    output_path = "/kaggle/working/"
else:
    data_path = "../../data/"
    output_path = "../../"

In [35]:
experiment_path = Path(data_path) / f"{data_pipeline}"
experiment_path.mkdir(parents=True, exist_ok=True)

In [36]:
ss = pd.read_csv("../../data/raw/sample_submission.csv")
target_column = ss.columns[-1]
target_column

'addicted_label'

In [37]:
X = ParquetFile(Path(data_path) / f"{input_pipeline}/train.parq").to_pandas()
X_test = ParquetFile(Path(data_path) / f"{input_pipeline}/test.parq").to_pandas()

X.info()

<class 'pandas.DataFrame'>
RangeIndex: 691369 entries, 0 to 691368
Data columns (total 12 columns):
 #   Column                   Non-Null Count   Dtype   
---  ------                   --------------   -----   
 0   age                      662440 non-null  float64 
 1   daily_screen_time_hours  595515 non-null  float64 
 2   social_media_hours       557374 non-null  float64 
 3   gaming_hours             564548 non-null  float64 
 4   work_study_hours         639851 non-null  float64 
 5   sleep_hours              646889 non-null  float64 
 6   notifications_per_day    623785 non-null  float64 
 7   app_opens_per_day        610659 non-null  float64 
 8   weekend_screen_time      579306 non-null  float64 
 9   gender                   662335 non-null  category
 10  stress_level             636221 non-null  float64 
 11  academic_work_impact     647145 non-null  float64 
dtypes: category(1), float64(11)
memory usage: 58.7 MB


In [38]:
X.head(100)

,age,daily_screen_time_hours,social_media_hours,gaming_hours,work_study_hours,sleep_hours,notifications_per_day,app_opens_per_day,weekend_screen_time,gender,stress_level,academic_work_impact
0,24.0,NaN,1.83,1.59,2.11,7.46,122.0,38.0,8.63,Male,1.0,0.0
1,19.0,5.97,1.08,NaN,3.03,8.22,76.0,19.0,NaN,Female,1.0,0.0
2,18.0,5.09,NaN,NaN,NaN,6.25,134.0,60.0,7.47,Female,0.0,1.0
3,21.0,6.42,1.26,1.42,3.36,8.85,112.0,94.0,8.66,Other,0.0,NaN
4,26.0,11.20,1.87,2.81,1.95,5.25,NaN,NaN,13.39,Female,1.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...
95,24.0,8.99,2.61,3.84,1.59,6.28,194.0,40.0,8.97,Other,2.0,1.0
96,35.0,3.25,1.40,0.88,0.58,7.00,NaN,NaN,4.64,Male,2.0,NaN
97,30.0,7.09,3.37,0.04,2.97,7.66,40.0,39.0,6.90,Male,1.0,0.0
98,20.0,3.51,0.92,0.42,1.97,7.71,111.0,162.0,8.17,Female,0.0,0.0


In [ ]:
drop_columns = X.columns
cat_columns = X.select_dtypes(include=['category']) 
num_columns = X.select_dtypes(include=['float64']) 

for frame in [X, X_test]:
    for col in num_columns:
        frame[f"{col}_round{place}"] = (frac * pow(10, digit)).apply(np.floor) % pow(10, digit-1)
    frame.drop(drop_columns, axis=1, inplace=True)

X.info()

<class 'pandas.DataFrame'>
RangeIndex: 691369 entries, 0 to 691368
Data columns (total 11 columns):
 #   Column                          Non-Null Count   Dtype  
---  ------                          --------------   -----  
 0   age_digit2                      662440 non-null  float64
 1   daily_screen_time_hours_digit2  595515 non-null  float64
 2   social_media_hours_digit2       557374 non-null  float64
 3   gaming_hours_digit2             564548 non-null  float64
 4   work_study_hours_digit2         639851 non-null  float64
 5   sleep_hours_digit2              646889 non-null  float64
 6   notifications_per_day_digit2    623785 non-null  float64
 7   app_opens_per_day_digit2        610659 non-null  float64
 8   weekend_screen_time_digit2      579306 non-null  float64
 9   stress_level_digit2             636221 non-null  float64
 10  academic_work_impact_digit2     647145 non-null  float64
dtypes: float64(11)
memory usage: 58.0 MB


In [40]:
X.head(100)

,age_digit2,daily_screen_time_hours_digit2,social_media_hours_digit2,gaming_hours_digit2,work_study_hours_digit2,sleep_hours_digit2,notifications_per_day_digit2,app_opens_per_day_digit2,weekend_screen_time_digit2,stress_level_digit2,academic_work_impact_digit2
0,0.0,NaN,3.0,9.0,0.0,6.0,0.0,0.0,3.0,0.0,0.0
1,0.0,6.0,8.0,NaN,2.0,2.0,0.0,0.0,NaN,0.0,0.0
2,0.0,8.0,NaN,NaN,NaN,5.0,0.0,0.0,6.0,0.0,0.0
3,0.0,1.0,6.0,1.0,5.0,4.0,0.0,0.0,6.0,0.0,NaN
4,0.0,9.0,7.0,1.0,5.0,5.0,NaN,NaN,9.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...
95,0.0,9.0,0.0,3.0,9.0,8.0,0.0,0.0,7.0,0.0,0.0
96,0.0,5.0,9.0,8.0,7.0,0.0,NaN,NaN,3.0,0.0,NaN
97,0.0,8.0,7.0,4.0,7.0,6.0,0.0,0.0,0.0,0.0,0.0
98,0.0,0.0,2.0,2.0,7.0,1.0,0.0,0.0,6.0,0.0,0.0


In [41]:
write(experiment_path / f"train.parq", X)
write(experiment_path / f"test.parq", X_test)